# Download an option chain, quotes and trades

Choose one underlying and trading date, filter its option contracts, then download the full quote and trade tapes for **every selected contract**.

Install this repository with `python -m pip install -e .` and select that Python environment as the notebook kernel (install `jupyterlab` and `ipykernel` if needed). Set `MASSIVE_TOKEN` in the launching environment; historical options access requires appropriate provider permissions. Set `OQ_DATA_ROOT` to an external data directory before starting the kernel, or configure it through the package configuration file. Never put credentials in this notebook.

Downloads use the existing cache: raw responses live under the configured raw location's `massive/` directory; contract-filter results use the processed location. Re-running reuses valid cached results. A whole chain can contain large tapes; this example downloads sequentially and retains only row counts in memory.


In [ ]:
from datetime import date
from time import monotonic

import polars as pl
from IPython.display import display

from oqdata.massive.client import MassiveClient
from oqdata.massive.options.contracts import REJECTED, ContractsFilter, filter_contracts
from oqdata.pool import PoolConfig

underlying = "SPY"
d = date(2024, 1, 2)
contracts_filter = ContractsFilter(max_dte=7, max_mns=2.0, min_opt_vol_k=1.0)
client = MassiveClient()
print("Raw cache:", client.locations.raw)
print("Processed cache:", client.locations.processed)


## Select contracts

`max_dte` counts **business days**, including zero-day expiries. `max_mns` limits absolute normalized log-moneyness relative to the adjusted previous close; it is not a percentage strike band. `min_opt_vol_k` sets the previous day's approximate cash-volume threshold in thousands of USD. Here `1.0` means $1,000.

The returned table also contains rejected contracts and their `reject_code`. Select `REJECTED.USED` before downloading. The filter checks today's quote availability; `drop_no_trades_today=False` keeps options with quotes but no trades. Half-day expiries are excluded here.


In [ ]:
contracts = filter_contracts(
    underlying, d,
    contracts_filter=contracts_filter,
    exclude_half_days=True,
    drop_no_trades_today=False,
    pool_config=PoolConfig.off(),
)
selected = contracts.filter(pl.col("reject_code") == REJECTED.USED)
print(f"{selected.height} of {contracts.height} contracts selected for {underlying} on {d}")
display(selected)


## Download every selected option

Both fetch methods paginate automatically and cache each option separately. These are **raw tapes**: `fetch_trades` retains closing prints and official-close conditions. For the standard condition and session filter, use `filter_valid_trades(option_ticker, d)` instead; it excludes CLOSE conditions by default, including when `within_session=False`.

An error is recorded for that option and the loop continues. Successful downloads remain cached, so rerunning retries missing downloads without repeating valid cached ones. Empty tapes are reported with zero rows.


In [ ]:
rows = []
started = monotonic()
option_tickers = selected["ticker"].to_list()
for i, option_ticker in enumerate(option_tickers, 1):
    row = {"ticker": option_ticker, "quote_rows": None, "trade_rows": None, "error": ""}
    errors = []
    for name, fetch in (("quote_rows", client.fetch_quotes), ("trade_rows", client.fetch_trades)):
        try:
            tape = fetch(option_ticker, d)
            row[name] = tape.height
            del tape
        except Exception as exc:
            errors.append(f"{name}: {type(exc).__name__}: {exc}")
    row["error"] = "; ".join(errors)
    rows.append(row)
    print(f"{i}/{len(option_tickers)} {option_ticker}: quotes={row['quote_rows']}, "
          f"trades={row['trade_rows']}, elapsed={monotonic() - started:.0f}s "
          f"{row['error']}", flush=True)

summary = pl.DataFrame(rows, schema={
    "ticker": pl.String, "quote_rows": pl.Int64, "trade_rows": pl.Int64, "error": pl.String,
})
display(summary)
print(f"Completed: {summary.filter(pl.col('error') == '').height}/{len(option_tickers)}")
display(summary.filter(pl.col("error") != ""))


## Read one cached option again

This uses the same client calls; valid cached tapes are reused. To apply the default trade-condition filter explicitly, replace the trade call below with `filter_valid_trades(option_ticker, d)` imported from `oqdata.massive.processing`.


In [ ]:
if option_tickers:
    option_ticker = option_tickers[0]
    quotes = client.fetch_quotes(option_ticker, d)
    trades = client.fetch_trades(option_ticker, d)
    display(quotes.head())
    display(trades.head())
